# Step 4 walkthrough: how the frozen split files are built, one piece at a time

This notebook runs the **real** split code (`src/cropyield/eval/splits.py`, `scripts/step4_build_splits.py`) on the
frozen `labels_v1`, in small cells. Every result stays in a variable, so you can `print()` anything.

**Rules of this notebook**
- It reads from the repository folder (found upward from the notebook by `pyproject.toml`) and **writes nothing** there. It never calls `step4.main()`.
- Two functions are copied into cells (`connected_clusters`, `assign_folds`) so you can print inside them.
  A check after each copy proves that it equals the real function.
- The last stage proves that the tables built here equal the 13 frozen files, byte for byte.
- Kernel: the Python of the repository's `.venv`. Run All takes about 10 seconds.

**The path, and the variable that holds each stage**

```text
Stage A  input            rows                      include_primary rows of one cohort (step4 line 49)
Stage B  distance         points, radius            lat/lon -> 3D points in km; 2 km -> a chord
Stage C  clusters         pairs -> graph -> groups  connected_clusters() unrolled
Stage D  folds            fold_of -> folds          assign_folds() unrolled
Stage E  four families    tables                    s1_cc2km, s2_region, random_diag, temporal
Stage F  the rules, live  leaks                     test fields within 2 km of a training field, per family
Stage G  proof            -                         the 13 files on disk equal what this notebook builds
Stage H  traps            -                         group_id vs cluster_id, row order
```

In [1]:
import importlib.util
import json
import sys
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist

from cropyield.eval import splits as S                    # the real split code
from cropyield.data.labels import LABELS_PATH, geodesic_km, sha256

# Load step 4's constants without running it: main() only runs under "python scripts/step4_build_splits.py".
spec = importlib.util.spec_from_file_location("step4", REPO / "scripts" / "step4_build_splits.py")
step4 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(step4)

SPLITS_DIR = REPO / "data" / "splits"
manifest = json.loads((SPLITS_DIR / "manifest.json").read_text(encoding="utf-8"))
pd.set_option("display.width", 200)

print("seed:", S.SEED, "| folds:", S.FOLDS)
print("s2_region unit per cohort:", step4.REGION_UNITS)
print("temporal:", step4.TEMPORAL)

seed: 20261029 | folds: 5
s2_region unit per cohort: {'malawi_2023': 5.0, 'kenya_oaf_2019_2020': 'admin_unit', 'zimbabwe_lacuna_2022_2023': 'admin_unit', 'zimbabwe_2024': 'kmeans20'}
temporal: {'kenya_oaf_2019_2020': (2019, 2020)}


## Stage A. The input: which rows get a split

`step4_build_splits.py` line 49: every `include_primary` row of the cohort, **with or without a chip**.
So a split never changes when the imagery coverage changes. Malawi 2022 has no included row, so it has no split file.

In [2]:
labels = pd.read_parquet(LABELS_PATH)
print("labels_v1 hash equals the one pinned in manifest.json:", sha256(LABELS_PATH) == manifest["labels_sha256"])
print(labels[labels["include_primary"]].groupby("cohort").size().rename("rows that get a split"))

COHORT = "kenya_oaf_2019_2020"
# Try also: "malawi_2023", "zimbabwe_lacuna_2022_2023", "zimbabwe_2024"

rows = labels[(labels["cohort"] == COHORT) & labels["include_primary"]].reset_index(drop=True)
FOLLOW = rows["obs_id"].iloc[0]
print(f"\n{COHORT}: {len(rows):,} rows. We follow {FOLLOW}")
rows[["obs_id", "season_year", "lat", "lon", "admin_unit", "cluster_2km", "yield_kg_ha"]].head()

labels_v1 hash equals the one pinned in manifest.json: True
cohort
kenya_oaf_2019_2020          3977
malawi_2023                   260
zimbabwe_2024                2458
zimbabwe_lacuna_2022_2023    1880
Name: rows that get a split, dtype: int64

kenya_oaf_2019_2020: 3,977 rows. We follow one_acre_fund_full:ee151220


,obs_id,season_year,lat,lon,admin_unit,cluster_2km,yield_kg_ha
0,one_acre_fund_full:ee151220,2019,0.113513,34.285943,alego,0,1583.2932
1,one_acre_fund_full:454485a8,2019,0.127618,34.314285,alego,1,4777.6567
2,one_acre_fund_full:37b3df46,2019,0.107713,34.291265,alego,0,0.0000
3,one_acre_fund_full:2274d106,2019,-0.046890,34.219426,alego,2,0.0000
4,one_acre_fund_full:d54cf69d,2019,-0.042607,34.195811,alego,2,0.0000


## Stage B. How the code measures 2 km

A KD-tree (a fast "who is near whom" index) measures **straight lines**. So `unit_xyz` turns each lat/lon into a
point in 3D, in km, with the centre of the Earth at (0, 0, 0). A straight line through the Earth is a **chord**.
`chord_km(2.0)` is the chord that belongs to 2 km along the ground. At this scale the two are almost the same.

In [3]:
lat, lon = rows["lat"].to_numpy(), rows["lon"].to_numpy()
points = S.unit_xyz(lat, lon)
radius = S.chord_km(2.0)
print("first 3 points (x, y, z in km):\n", points[:3].round(3))
print(f"\n2 km along the ground = {radius:.9f} km as a chord")

straight = np.linalg.norm(points[0] - points[1])
ground = geodesic_km(lat[0], lon[0], lat[1], lon[1])
print(f"rows 0 and 1: chord {straight:.6f} km, along the ground {ground:.6f} km")

first 3 points (x, y, z in km):
 [[5263.95  3588.931   12.622]
 [5262.171 3591.533   14.19 ]
 [5263.617 3589.421   11.977]]

2 km along the ground = 1.999999992 km as a chord
rows 0 and 1: chord 3.520245 km, along the ground 3.520245 km


## Stage C. Clusters: `connected_clusters()` unrolled (`splits.py` line 35)

1. Find **every pair** of fields closer than 2 km.
2. Treat each pair as a link in a graph.
3. A cluster is a **connected component**: all fields you can reach by walking along links.

So a cluster is a **chain**. A and C can be 5 km apart and still share a cluster, if B sits within 2 km of both.
That is why a cluster is not a surveyed village.

In [4]:
pairs = cKDTree(points).query_pairs(r=radius, output_type="ndarray")
print(f"{len(pairs):,} pairs of fields closer than 2 km. First 5 (row numbers):\n", pairs[:5])

n = len(points)
if len(pairs) == 0:
    groups = np.arange(n)
else:
    graph = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(n, n))
    groups = connected_components(graph, directed=False)[1]

assert np.array_equal(groups, S.connected_clusters(lat, lon, 2.0))
print("OK: equals S.connected_clusters().")

10,508 pairs of fields closer than 2 km. First 5 (row numbers):
 [[1093 1114]
 [1093 1113]
 [1093 1095]
 [1093 1094]
 [1093 1792]]
OK: equals S.connected_clusters().


In [5]:
size = pd.Series(groups).value_counts()
print(f"{len(size):,} clusters | {int((size == 1).sum()):,} are a single field | largest has {size.iloc[0]:,} fields")
print("cluster size quantiles:", size.quantile([0.5, 0.9, 0.99, 1.0]).to_dict())

big = np.flatnonzero(groups == size.index[0])
span = pdist(points[big]).max() if len(big) > 1 else 0.0
print(f"the largest cluster spans {span:.1f} km from end to end: a chain, not a village")

mine = groups[rows.index[rows["obs_id"] == FOLLOW][0]]
print(f"\nour field {FOLLOW} is in cluster {mine}, with {int((groups == mine).sum())} fields")

849 clusters | 281 are a single field | largest has 168 fields
cluster size quantiles: {0.5: 2.0, 0.9: 9.0, 0.99: 39.039999999999964, 1.0: 168.0}
the largest cluster spans 24.6 km from end to end: a chain, not a village

our field one_acre_fund_full:ee151220 is in cluster 0, with 3 fields


## Stage D. Folds: `assign_folds()` unrolled (`splits.py` line 59)

1. Count the fields per group.
2. Shuffle the groups with the seed. This only decides the order of groups that have **the same size**.
3. Take the **largest group first**, and put it into the fold that has the **fewest fields so far**. Repeat.

Result: whole groups go to one fold, and the 5 folds have almost the same number of fields.

In [6]:
k, seed = S.FOLDS, S.SEED
sizes = pd.Series(groups).value_counts()
order = np.array(sizes.index, dtype=object)
np.random.default_rng(seed).shuffle(order)

fold_of, filled = {}, np.zeros(k)
for step, group in enumerate(sorted(order, key=lambda g: -sizes[g])):   # sorted is stable: ties stay shuffled
    fold_of[group] = int(np.argmin(filled))
    filled[fold_of[group]] += sizes[group]
    if step < 8:
        print(f"step {step}: group {group:>4} ({sizes[group]:>3} fields) -> fold {fold_of[group]}   fields per fold now {filled.astype(int).tolist()}")
folds = np.array([fold_of[group] for group in groups])

print("\nfinal fields per fold:", filled.astype(int).tolist())
assert np.array_equal(folds, S.assign_folds(groups))
print("OK: equals S.assign_folds().")

step 0: group   81 (168 fields) -> fold 0   fields per fold now [168, 0, 0, 0, 0]
step 1: group  132 (125 fields) -> fold 1   fields per fold now [168, 125, 0, 0, 0]
step 2: group   75 ( 96 fields) -> fold 2   fields per fold now [168, 125, 96, 0, 0]
step 3: group  199 ( 72 fields) -> fold 3   fields per fold now [168, 125, 96, 72, 0]
step 4: group  157 ( 64 fields) -> fold 4   fields per fold now [168, 125, 96, 72, 64]
step 5: group  117 ( 59 fields) -> fold 4   fields per fold now [168, 125, 96, 72, 123]
step 6: group  253 ( 45 fields) -> fold 3   fields per fold now [168, 125, 96, 117, 123]
step 7: group  209 ( 42 fields) -> fold 2   fields per fold now [168, 125, 138, 117, 123]

final fields per fold: [796, 796, 795, 795, 795]
OK: equals S.assign_folds().


## Stage E. The four families (`step4.cohort_splits`, real functions)

| Family | Group = the unit held out together | Folds |
|---|---|---|
| `s1_cc2km` new village | 2 km clusters (Stage C) | 5 |
| `s2_region` new region | `REGION_UNITS`: Kenya and Lacuna district (`admin_unit`), Malawi 5 km clusters, Zimbabwe 2024 20 k-means blocks | 5 |
| `random_diag` | every field alone | 5, random |
| `temporal` (Kenya only) | district; repeat 0 = train 2019 / test 2020, repeat 1 = the reverse | `train` / `test` |

In [7]:
tables = {table["family"].iloc[0]: table for table in step4.cohort_splits(rows, COHORT)}
for family, table in tables.items():
    first = table[table["repeat"] == 0]
    print(f"{family:12s} rows {len(table):>5} | repeats {table['repeat'].nunique()} | groups {first['group_id'].nunique():>4} "
          f"| fields per fold {first.groupby('fold').size().to_dict()}")

s1 = tables["s1_cc2km"]
assert (s1["group_id"].to_numpy() == groups.astype(str)).all() and (s1["fold"].to_numpy() == folds.astype(str)).all()
print("\nOK: s1_cc2km uses exactly the groups and folds of Stages C and D.")
print("\nour field in each family (temporal has one row per repeat):")
print(pd.concat([table[table["obs_id"] == FOLLOW] for table in tables.values()]).fillna("").to_string(index=False))

s1_cc2km     rows  3977 | repeats 1 | groups  849 | fields per fold {'0': 796, '1': 796, '2': 795, '3': 795, '4': 795}


s2_region    rows  3977 | repeats 1 | groups   50 | fields per fold {'0': 800, '1': 798, '2': 797, '3': 794, '4': 788}
random_diag  rows  3977 | repeats 1 | groups 3977 | fields per fold {'0': 796, '1': 796, '2': 795, '3': 795, '4': 795}
temporal     rows  7954 | repeats 2 | groups   50 | fields per fold {'test': 2209, 'train': 1768}

OK: s1_cc2km uses exactly the groups and folds of Stages C and D.

our field in each family (temporal has one row per repeat):
                     obs_id              cohort  season_year  cluster_id group_id  fold  repeat      family test_subset
one_acre_fund_full:ee151220 kenya_oaf_2019_2020         2019           0        0     2       0    s1_cc2km            
one_acre_fund_full:ee151220 kenya_oaf_2019_2020         2019           0    alego     3       0   s2_region            
one_acre_fund_full:ee151220 kenya_oaf_2019_2020         2019           0        0     4       0 random_diag            
one_acre_fund_full:ee151220 kenya_oaf_2019_2020        

In [8]:
problems = [p for table in tables.values() for p in S.check_split(table, rows)]
print("check_split problems:", problems if problems else "none (every row once per repeat, no group in two folds, s1: no test field within 2 km of training)")

check_split problems: none (every row once per repeat, no group in two folds, s1: no test field within 2 km of training)


## Stage F. The rules, measured live: how close is a test field to a training field?

Only `s1_cc2km` promises a 2 km gap. `splits.py` says it in its first lines: "There is no distance buffer here".
This cell counts, per family, the test fields that have a training field **within 2 km**.
These are the numbers to discuss with the team.

In [9]:
def near_training(table, km=2.0):
    '''Per repeat and test fold: test fields with a training field closer than km, and the smallest gap.'''
    where = rows.set_index("obs_id").loc[table["obs_id"]]
    t_lat, t_lon = where["lat"].to_numpy(), where["lon"].to_numpy()
    out = []
    for repeat in sorted(table["repeat"].unique()):
        block = (table["repeat"] == repeat).to_numpy()
        fold = table["fold"].to_numpy()
        tests = ["test"] if "test" in set(fold[block]) else sorted(set(fold[block]))
        for f in tests:
            test = block & (fold == f)
            train = block & (fold == "train") if f == "test" else block & (fold != f)
            gap = S.nearest_km(t_lat, t_lon, np.flatnonzero(test), np.flatnonzero(train))
            out.append({"repeat": repeat, "test fold": f, "test fields": int(test.sum()),
                        f"within {km:g} km of training": int((gap < km).sum()), "smallest gap km": round(float(gap.min()), 3),
                        "median gap km": round(float(np.median(gap)), 2)})
    return pd.DataFrame(out)

for family, table in tables.items():
    result = near_training(table)
    total = result[f"within 2 km of training"].sum()
    print(f"\n### {family}: {total:,} test fields within 2 km of a training field")
    print(result.to_string(index=False))


### s1_cc2km: 0 test fields within 2 km of a training field
 repeat test fold  test fields  within 2 km of training  smallest gap km  median gap km
      0         0          796                        0            2.004           3.26
      0         1          796                        0            2.005           3.26
      0         2          795                        0            2.006           3.48
      0         3          795                        0            2.003           3.46
      0         4          795                        0            2.003           3.44

### s2_region: 806 test fields within 2 km of a training field
 repeat test fold  test fields  within 2 km of training  smallest gap km  median gap km
      0         0          800                      220            0.012           3.52
      0         1          798                      223            0.018           4.00
      0         2          797                      120            0.003           


### temporal: 1,757 test fields within 2 km of a training field
 repeat test fold  test fields  within 2 km of training  smallest gap km  median gap km
      0      test         2209                      833            0.001           2.79
      1      test         1768                      924            0.001           1.92


In [10]:
if "temporal" in tables:
    t = tables["temporal"]
    print("temporal: which test districts were seen in the training year?")
    print(t[t["fold"] == "test"].groupby(["repeat", "test_subset"]).size().unstack())
else:
    print(f"{COHORT} has no temporal split: only Kenya has two years with imagery.")

temporal: which test districts were seen in the training year?
test_subset  new_region  seen_region
repeat                              
0                 360.0       1849.0
1                   NaN       1768.0


## Stage G. Proof: this notebook builds exactly the 13 frozen files

`step4_build_splits.py` writes `split.to_csv(index=False, lineterminator="\n")`, and **refuses** to overwrite a file
whose text would change (line 58). We build the same text here, for every cohort and family, and compare it with
the file on disk and with the hash in `manifest.json`.

In [11]:
proof = []
for cohort in step4.REGION_UNITS:
    cohort_rows = labels[(labels["cohort"] == cohort) & labels["include_primary"]].reset_index(drop=True)
    for table in step4.cohort_splits(cohort_rows, cohort):
        path = SPLITS_DIR / f"{cohort}__{table['family'].iloc[0]}.csv"
        text = table.to_csv(index=False, lineterminator="\n")
        proof.append({"file": path.name, "rows": len(table),
                      "same text as the file": path.read_text(encoding="utf-8") == text,
                      "file hash = manifest": sha256(path) == manifest["files"][path.name]["sha256"]})
proof = pd.DataFrame(proof)
print(proof.to_string(index=False))
assert proof["same text as the file"].all() and proof["file hash = manifest"].all() and len(proof) == 13
print("\nOK: all 13 split files equal what this notebook builds, and their hashes equal manifest.json.")

                                      file  rows  same text as the file  file hash = manifest
                 malawi_2023__s1_cc2km.csv   260                   True                  True
                malawi_2023__s2_region.csv   260                   True                  True
              malawi_2023__random_diag.csv   260                   True                  True
         kenya_oaf_2019_2020__s1_cc2km.csv  3977                   True                  True
        kenya_oaf_2019_2020__s2_region.csv  3977                   True                  True
      kenya_oaf_2019_2020__random_diag.csv  3977                   True                  True
         kenya_oaf_2019_2020__temporal.csv  7954                   True                  True
   zimbabwe_lacuna_2022_2023__s1_cc2km.csv  1880                   True                  True
  zimbabwe_lacuna_2022_2023__s2_region.csv  1880                   True                  True
zimbabwe_lacuna_2022_2023__random_diag.csv  1880            

## Stage H. Two traps when you write a model

1. **Group on `group_id`, never on `cluster_id`.** `cluster_id` is `labels_v1.cluster_2km`. It was computed on
   **all** rows of the cohort (also excluded rows and the 2022 year). `group_id` was computed on the included rows only.
   Two included fields can be linked only through an excluded field: then they share a `cluster_id`, but not a `group_id`.
2. **Join on `obs_id`, never by row position.** A split file is in `labels_v1` order. `load_xy` sorts by `obs_id`
   and leaves out rows with fewer than 4 clear months.

In [12]:
for cohort in step4.REGION_UNITS:
    split = pd.read_csv(SPLITS_DIR / f"{cohort}__s1_cc2km.csv", dtype=str)
    crossing = (split.groupby("cluster_id")["fold"].nunique() > 1).sum()
    print(f"{cohort:28s} group_id values {split['group_id'].nunique():>5} | cluster_id values {split['cluster_id'].nunique():>5} "
          f"| cluster_id values found in 2+ folds: {crossing}")

split = pd.read_csv(SPLITS_DIR / f"{COHORT}__s1_cc2km.csv", dtype=str)
print("\nsplit file sorted by obs_id?", split["obs_id"].is_monotonic_increasing, "(load_xy output is sorted)")

malawi_2023                  group_id values    53 | cluster_id values    53 | cluster_id values found in 2+ folds: 0
kenya_oaf_2019_2020          group_id values   849 | cluster_id values   849 | cluster_id values found in 2+ folds: 0
zimbabwe_lacuna_2022_2023    group_id values   289 | cluster_id values   272 | cluster_id values found in 2+ folds: 12
zimbabwe_2024                group_id values   996 | cluster_id values   956 | cluster_id values found in 2+ folds: 27

split file sorted by obs_id? False (load_xy output is sorted)


## The split rules, in one table: what the team must agree on

| # | Rule | Where in the code | Why | Open point for the team |
|---:|---|---|---|---|
| 1 | Every `include_primary` row gets a split, with or without a chip | `step4_build_splits.py` line 49 | Coverage changes never change a split | – |
| 2 | Seed 20261029, 5 folds, largest group into the emptiest fold | `splits.py` lines 18–19, 59 | Balanced folds, same files on every machine | – |
| 3 | `s1_cc2km`: no test field within 2 km of a training field | `splits.py` lines 78, 129–135 | Stops neighbour fields from leaking | Kenya yield stays linked up to about 50 km: 2 km gives separation, not independence |
| 4 | `s2_region`: whole districts (Kenya, Lacuna), 5 km clusters (Malawi), 20 k-means blocks (Zimbabwe 2024) | `step4_build_splits.py` line 26 | Tests a new region | **No buffer** (Stage F count). Drop near training rows in the model runner? |
| 5 | `temporal`: Kenya 2019 → 2020 and back, `test_subset` seen / new district | `splits.py` line 99 | Tests a new season | **Not blocked in space** (Stage F count). Report seen and new apart |
| 6 | `random_diag`: diagnostic only | `splits.py` line 114 | Shows the optimism of random splits | Never in the abstract |
| 7 | Files are frozen: step 4 refuses a changed file; `manifest.json` pins the hashes | `step4_build_splits.py` line 58 | Splits come before models | Step 4 rewrites `manifest.json` on every run (V-21). Only step 8 stops on a mismatch |
| 8 | Fold 0 of `s1_cc2km` is spent: no new arm is scored on it | owner ruling, 1 Nov | Keeps one clean test for the end | Every new runner must skip fold 0 |
| 9 | Models group on `group_id` and join on `obs_id`; inner CV uses the same groups | Stage H | No leak inside the training fold | – |
| 10 | Per-fold scores only with ≥ 100 test fields and ≥ 10 groups | `docs/EVALUATION.md` | Small folds are noise | Malawi: pooled scores only |